# Continuous Evaluation

## Objective
The evaluation layer of Agent Ops Level 1, in the two modes GEAP supports:
- **Offline evaluation** (this notebook): score the deployed agent against a curated golden evalset with a managed, console-tracked evaluation run (`client.evals.create_evaluation_run`). The run drives the agent, scores the metric menu, and appears in the GEAP Evaluation dashboard with cross-version lineage. Requires the agent deployed in NB1.
- **Online evaluation**: an online monitor over live traffic, plus the Cloud Monitoring drift alert that fires the loop.

Documentation: [Evaluate your agents](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/evaluate-agents) · [Evaluate offline](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/evaluate-offline) · [Manage metrics](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/manage-metrics) · [View results](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/view-results)

## The Metric Menu

This agent is single-turn with no tools, so we score it with:
- **Final Response Quality**: a custom `LLMMetric` (each violation cites the correct brand rule; the fix is actionable), registered in the console Metric Registry. The headline metric.
- **Safety** (`types.RubricMetric.SAFETY`) and **General Quality** (`types.RubricMetric.GENERAL_QUALITY`): predefined.

The predefined Agent *Final Response Quality* / *Hallucination* raters need a tool trajectory or grounding context this agent doesn't have, so the headline is our own rubric.

In [1]:
import os, json, time, importlib.util
import pandas as pd
from vertexai import Client, types
from google.genai import types as genai_types

PROJECT = 'sandbox-401718'          # @param {type:"string"}
AGENT = 'brand-guidelines-checker'  # @param {type:"string"}
# eval client region — match the agent's region so the run shows in the regional console
EVAL_LOCATION = 'us-central1'  # @param {type:"string"}
MODEL = 'gemini-2.5-flash'     # @param {type:"string"}
AGENT_DIR = f'../agents/{AGENT}'

os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'true'
os.environ['GOOGLE_CLOUD_LOCATION'] = (
    'global'  # agent's Gemini endpoint (agent_info only)
)
os.environ['GOOGLE_CLOUD_PROJECT'] = PROJECT
os.environ['MODEL'] = MODEL

# v1beta1 for the Preview evals surface.
client = Client(
    project=PROJECT,
    location=EVAL_LOCATION,
    http_options=genai_types.HttpOptions(api_version='v1beta1'),
)

# Load the local agent — only for agent_info metadata the raters use.
spec = importlib.util.spec_from_file_location(
    'agent_under_test', f'{AGENT_DIR}/app/agent.py'
)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
root_agent = mod.root_agent
print(open(f'{AGENT_DIR}/eval/criteria.json').read())


# Quiet noise: google-genai's async client logs harmless teardown errors in notebooks
# (does not affect results). ADK's [EXPERIMENTAL] eval-feature warnings are left visible.
import warnings, logging
warnings.filterwarnings('ignore', message='.*was never awaited.*')
logging.getLogger('asyncio').setLevel(logging.CRITICAL)

/opt/conda/lib/python3.10/site-packages/google/cloud/aiplatform/models.py:52: FutureWarning: Support for google-cloud-storage < 3.0.0 will be removed in a future version of google-cloud-aiplatform. Please upgrade to google-cloud-storage >= 3.0.0.
  from google.cloud.aiplatform.utils import gcs_utils
/opt/conda/lib/python3.10/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_version" in LlmResponse has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_version" in Event has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


{
  "_comment": "ADK EvalConfig — the offline 'good enough?' threshold check. eval_tool/run_eval.py executes these via ADK's LocalEvalService (the engine adk eval / agents-cli eval run wrap), which surfaces the GEAP Agent Evaluation raters locally. The brand checker is single-turn with no tools, so the applicable GEAP metrics are Final Response Quality (rubric_based_final_response_quality_v1 — the headline the threshold's bootstrap CI runs on), Hallucination (hallucinations_v1 — factuality guardrail), and Safety (safety_v1 — policy guardrail). GEAP's Task Success is a multi-turn rater and does not apply here. All are real Gemini LLM judges and need GOOGLE_CLOUD_PROJECT / GOOGLE_CLOUD_LOCATION.",
  "criteria": {
    "rubric_based_final_response_quality_v1": {
      "threshold": 0.8,
      "judge_model_options": { "judge_model": "gemini-2.5-flash", "num_samples": 5 },
      "rubrics": [
        { "rubric_id": "cites_rule",     "rubric_content": { "textProperty": "Each reported violation 

/opt/conda/lib/python3.10/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_code" in LlmAgentConfig has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


## The Golden Evalset

The curated regression set the eval scores against — realistic inputs paired with the expected verdict and the rule each violation should cite. The loop grows it from real production failures (see `loop/README.md`).

In [2]:
pd.set_option('display.max_colwidth', None)
golden = json.load(open(f'{AGENT_DIR}/eval/golden.evalset.json'))
cases = golden['eval_cases']
golden_df = pd.DataFrame(
    [
        {
            'id': c['id'],
            'verdict': c['expected']['verdict'],
            'cites_rules': [v['rule'] for v in c['expected'].get('violations', [])],
            'input': c['input'],
        }
        for c in cases
    ]
)
print(f'golden evalset: {len(golden_df)} cases')
golden_df

golden evalset: 20 cases


,id,verdict,cites_rules,input
0,hype-word-fail,FAIL,[1],Our revolutionary new blender is best-in-class.
1,clean-copy-pass,PASS,[],"You'll love how quiet this blender is. On sale now, while supplies last."
2,discount-missing-disclaimer,FAIL,[3],Save big on every coffee maker this weekend.
3,guaranteed-savings-fail,FAIL,[2],Guaranteed to save you $50 on your first order.
4,all-caps-headline-fail,FAIL,[4],BIGGEST SALE EVER on kitchen essentials.
5,consumers-wording-fail,FAIL,[5],Consumers trust our cookware every day.
6,plain-pass,PASS,[],Your morning just got easier with our new drip coffee maker.
7,warm-tone-pass,PASS,[],We think you'll enjoy the gentle hum of this quiet blender.
8,no-claims-pass,PASS,[],This kettle heats water fast and keeps it warm for you.
9,sentence-case-pass,PASS,[],Our new cookware set is here for you.


## Run a Console-Tracked Evaluation

Register the custom rubric, then `create_evaluation_run` against the deployed agent (NB1). The run drives the agent over the golden inputs, scores the menu (custom + predefined), and lands in the console Evaluation dashboard. `run.show()` renders the report here too.

This makes real Gemini judge calls and creates a managed run (async — we poll until it finishes). Requires NB1's deployed agent; writes results to a GCS output path.

Note: console visibility for Evaluation Experiments may be limited (Preview).

Note: a console-tracked run reads the deployed agent's prompt/response from its traces, which export to Cloud Trace asynchronously. Right after a redeploy the trace pipeline is cold, so some cases may come back unscored on the first run; coverage fills in as you re-run or after the agent has been invoked a few times. For guaranteed full-coverage scoring, use the in-process path (`run_inference` + `evaluate` on the local agent, as NB3 does) — at the cost of no console run.

Documentation: [Evaluate your agents](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/evaluate-agents) · [Manage metrics](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/manage-metrics) · [View results](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/view-results)

In [3]:
from google.cloud import storage
import vertexai
from vertexai import agent_engines

# 1) Find the DEPLOYED agent (from NB1) — the run drives it.
vertexai.init(project=PROJECT, location=EVAL_LOCATION)
DISPLAY_NAME = 'Brand Guidelines Checker'  # @param {type:"string"}
deployed = next(
    (a for a in agent_engines.list() if a.display_name == DISPLAY_NAME), None
)
assert (
    deployed is not None
), f"No deployed agent '{DISPLAY_NAME}'. Run NB1 (01-deploy-agent) first."
AGENT_RESOURCE = deployed.resource_name
print('deployed agent:', AGENT_RESOURCE)

# 2) Ensure a GCS output bucket for the run.
DEST_BUCKET = (
    f'{PROJECT}-agent-staging'  # single GCS bucket for the demo (NB1 created it)
)
sc = storage.Client(project=PROJECT)
if sc.lookup_bucket(DEST_BUCKET) is None:
    sc.create_bucket(DEST_BUCKET, location=EVAL_LOCATION)
    print('created bucket:', DEST_BUCKET)

# 3) Define + register the custom Final Response Quality rubric (idempotent: reuse an
#    existing metric with the same display name instead of creating a duplicate each run).
final_response_quality = types.LLMMetric(
    name='final_response_quality',
    prompt_template=(
        'You are grading a single-turn brand-guidelines checker.\n'
        'Marketing copy:\n{prompt}\n\n'
        "Checker's JSON verdict:\n{response}\n\n"
        'Top score only if every reported violation cites the correct brand rule id '
        'and the suggested_fix is concrete and would resolve it.\n'
        'Return ONLY JSON: {"score": <float 0-1>, "explanation": "<one sentence>"}'
    ),
    result_parsing_function=(
        'import json, re\n'
        'def parse_results(responses):\n'
        "    m = re.search(r'\\{.*\\}', responses[0], re.S)\n"
        '    d = json.loads(m.group(0)) if m else {}\n'
        "    return {'score': float(d.get('score', 0.0)), 'explanation': d.get('explanation', '')}\n"
    ),
)
_existing = next(
    (
        m
        for m in (client.evals.list_evaluation_metrics().evaluation_metrics or [])
        if m.display_name == final_response_quality.name
    ),
    None,
)
if _existing:
    metric_resource = _existing.name
    print('reusing existing metric:', metric_resource)
else:
    metric_resource = str(
        client.evals.create_evaluation_metric(metric=final_response_quality)
    )
    print('created metric:', metric_resource)
registered_frq = types.Metric(
    name='final_response_quality', metric_resource_name=metric_resource
)

# 4) Capture responses SYNCHRONOUSLY with run_inference (full coverage — avoids the
#    async trace-export lag you get when create_evaluation_run does its own inference).
prompts = pd.DataFrame(
    [
        {
            'prompt': c['input'],
            'session_inputs': types.evals.SessionInput(
                user_id='eval-user', app_name=AGENT
            ),
        }
        for c in cases
    ]
)
inferred = client.evals.run_inference(agent=AGENT_RESOURCE, src=prompts)

# 5) Score the PRE-INFERRED dataset (no agent= / agent_info — the responses are already
#    captured, so the run scores all cases and the custom metric computes reliably).
run = client.evals.create_evaluation_run(
    dataset=inferred,
    metrics=[
        registered_frq,
        types.RubricMetric.SAFETY,
        types.RubricMetric.GENERAL_QUALITY,
    ],
    dest=f'gs://{DEST_BUCKET}/eval-runs',
    display_name=f'{AGENT}-offline-eval',
)
print('created run:', run.name)
while not any(s in str(run.state) for s in ('SUCCEEDED', 'FAILED', 'CANCELLED')):
    time.sleep(15)
    run = client.evals.get_evaluation_run(name=run.name, include_evaluation_items=True)
print('state:', run.state)

# 6) The report — also visible in the console Evaluation dashboard.
run.show()

deployed agent: projects/757654702990/locations/us-central1/reasoningEngines/3538275147427348480


/var/tmp/ipykernel_716799/647610601.py:49: ExperimentalWarning: The Vertex SDK GenAI evals.list_evaluation_metrics module is experimental, and may change in future versions.
  for m in (client.evals.list_evaluation_metrics().evaluation_metrics or [])


reusing existing metric: projects/757654702990/locations/us-central1/evaluationMetrics/8010566730952736768


Agent Run: 100%|██████████| 20/20 [00:48<00:00,  2.45s/it]
/var/tmp/ipykernel_716799/647610601.py:83: ExperimentalWarning: The Vertex SDK GenAI evals.create_evaluation_run module is experimental, and may change in future versions.
  run = client.evals.create_evaluation_run(
/opt/conda/lib/python3.10/site-packages/vertexai/_genai/_evals_common.py:2978: ExperimentalWarning: The Vertex SDK GenAI evals.create_evaluation_item module is experimental, and may change in future versions.
  eval_item = evals_module.create_evaluation_item(
/opt/conda/lib/python3.10/site-packages/vertexai/_genai/_evals_common.py:2985: ExperimentalWarning: The Vertex SDK GenAI evals.create_evaluation_set module is experimental, and may change in future versions.
  evaluation_set = evals_module.create_evaluation_set(


created run: projects/757654702990/locations/us-central1/evaluationRuns/3666475454446960640


/var/tmp/ipykernel_716799/647610601.py:96: ExperimentalWarning: The Vertex SDK GenAI evals.get_evaluation_run module is experimental, and may change in future versions.
  run = client.evals.get_evaluation_run(name=run.name, include_evaluation_items=True)
/opt/conda/lib/python3.10/site-packages/vertexai/_genai/_evals_common.py:2759: ExperimentalWarning: The Vertex SDK GenAI evals.get_evaluation_set method is experimental, and may change in future versions.
  eval_set = evals_module.get_evaluation_set(
/opt/conda/lib/python3.10/site-packages/vertexai/_genai/_evals_common.py:2766: ExperimentalWarning: The Vertex SDK GenAI evals.get_evaluation_item method is experimental, and may change in future versions.
  evals_module.get_evaluation_item(name=item_name)


state: EvaluationRunState.SUCCEEDED


## How Evaluation Feeds the Broader Feedback Loop

The Quality Flywheel scores each candidate and runs `eval_tool/compare.py`: a candidate is "good enough" if its Final Response Quality matches the live baseline within a bootstrap confidence interval, with zero regressions on existing golden cases. The loop creates its own baseline on its first step — it scores undeployed candidates locally, so it uses the local/`evaluate` path rather than console-tracked runs. See `loop/README.md`.

## *TODO - online monitor SCREENSHOT*

## Online Evaluation — Continuous Monitoring

An online monitor scores a sample of live traffic continuously: every ~10 min it samples Cloud Trace/Logging, scores via the Evaluation Service, and writes results to Cloud Logging + Cloud Monitoring + the Evaluation dashboard (and per-trace). Its only prerequisite is the agent's prompt/response capture — already enabled in NB1.

Today it's console-only (Preview): *Agent Platform → Agents → Evaluation → Online monitors → New monitor* — pick the agent, set a sampling %, and choose metrics (Safety + General Quality, or the custom Final Response Quality from the Metric Registry; the predefined Agent Final Response Quality rater needs a tool trajectory this single-turn agent lacks).

The SDK analog is incoming. Once it lands, this section will mirror the offline flow — roughly:
```python
# not yet available — the managed online-monitor analog of create_evaluation_run
client.evals.create_online_monitor(
    agent=AGENT_RESOURCE, sampling_percentage=5,
    metrics=[types.RubricMetric.SAFETY, types.RubricMetric.GENERAL_QUALITY, registered_frq],
)
```

The drift alert that turns this signal into the feedback-loop trigger lives in NB3 (Feedback Loop Trigger).

Documentation: [Evaluate online](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/evaluate-online)